# Workshop on signal analysis and feature extraction

Course: Intelligent Sensing and Sense Making

Website: https://www.iss.nus.edu.sg/executive-education/course/detail/intelligent-sensing-and-sense--making/artificial-intelligence

Contact: Tian Jing

Email: tianjing@nus.edu.sg

**Objective**: In this workshop, we will extract statistical features from wavelet coefficients from human wearable sensor data, and then perform classification for human activity classification

**Reference**: PyWavelets - Wavelet Transforms in Python, https://github.com/PyWavelets/pywt. The Help document of this tool is provided at https://pywavelets.readthedocs.io/en/latest/


**Dataset**: UCI Human Activity Recognition Using Smartphones. This dataset contains sensor data for 30 persons, each person performed six activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) wearing a smartphone (Samsung Galaxy S II) on the waist. Using its embedded accelerometer and gyroscope, 3-axial linear acceleration and 3-axial angular velocity are recorded at a constant rate of 50Hz. Website: https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones

In [1]:
import sys
!{sys.executable} -m pip install PyWavelets

In [2]:
# Load necessary packages
import os
import numpy as np
import matplotlib.pyplot as plt
import pywt
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix


In [3]:
# Grant access to google drive.
# Run this cell, then you’ll see a link, click on that link, allow access
# Copy the code that pops up, paste it in the box, hit Enter

# Local execution replaces the Google Colab/Drive mount.
from pathlib import Path
local_candidates = [Path.cwd(), Path(r'C:\\Users\\Setekh\\Desktop\\ISY5002\\Part3\\Day2\\ISSM Day2 workshop\\Day2')]
local_day2 = next((p for p in local_candidates if (p / 'data' / 'UCI_HAR').exists()), None)
if local_day2 is None:
    raise FileNotFoundError('Cannot find the local Day2/data/UCI_HAR folder.')
os.chdir(local_day2)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))


Working directory: C:\Users\Setekh\Desktop\ISY5002\Part3\Day2\ISSM Day2 workshop\Day2
Files: ['data', 'ISSM_Day2_wk_signal_WangMingyang.ipynb', 'ISSM_Day2_wk_signal_yourname.ipynb']


In [4]:
# Load training data and label, test data and label

def read_signals(filename):
    with open(filename, 'r') as fp:
        data = fp.read().splitlines()
        data = map(lambda x: x.rstrip().lstrip().split(), data)
        data = [list(map(float, line)) for line in data]
    return data

def read_labels(filename):
    with open(filename, 'r') as fp:
        activities = fp.read().splitlines()
        activities = list(map(int, activities))
    return activities

def randomize(dataset, labels):
    permutation = np.random.permutation(labels.shape[0])
    shuffled_dataset = dataset[permutation, :, :]
    shuffled_labels = labels[permutation]
    return shuffled_dataset, shuffled_labels

INPUT_FOLDER_TRAIN = 'data/UCI_HAR/train/Inertial Signals/'
INPUT_FOLDER_TEST = 'data/UCI_HAR/test/Inertial Signals/'

INPUT_FILES_TRAIN = ['body_acc_x_train.txt', 'body_acc_y_train.txt', 'body_acc_z_train.txt',
                     'body_gyro_x_train.txt', 'body_gyro_y_train.txt', 'body_gyro_z_train.txt',
                     'total_acc_x_train.txt', 'total_acc_y_train.txt', 'total_acc_z_train.txt']

INPUT_FILES_TEST = ['body_acc_x_test.txt', 'body_acc_y_test.txt', 'body_acc_z_test.txt',
                     'body_gyro_x_test.txt', 'body_gyro_y_test.txt', 'body_gyro_z_test.txt',
                     'total_acc_x_test.txt', 'total_acc_y_test.txt', 'total_acc_z_test.txt']

LABELFILE_TRAIN = 'data/UCI_HAR/train/y_train.txt'
LABELFILE_TEST = 'data/UCI_HAR/test/y_test.txt'

train_signals, test_signals = [], []

for input_file in INPUT_FILES_TRAIN:
    signal = read_signals(INPUT_FOLDER_TRAIN + input_file)
    train_signals.append(signal)
train_signals = np.transpose(np.array(train_signals), (1, 2, 0))

for input_file in INPUT_FILES_TEST:
    signal = read_signals(INPUT_FOLDER_TEST + input_file)
    test_signals.append(signal)
test_signals = np.transpose(np.array(test_signals), (1, 2, 0))

train_labels = read_labels(LABELFILE_TRAIN)
test_labels = read_labels(LABELFILE_TEST)

print("The train dataset contains %d records with %d length (# of data points) and %d components (# of sensors)." % (train_signals.shape[0], train_signals.shape[1], train_signals.shape[2]))
print("The test dataset contains %s records with %d length (# of data points) and %d components (# of sensors)." % (test_signals.shape[0], test_signals.shape[1], test_signals.shape[2]))

uci_har_signals_train, uci_har_labels_train = randomize(train_signals, np.array(train_labels))
uci_har_signals_test, uci_har_labels_test = randomize(test_signals, np.array(test_labels))

activity_label = ['WALKING', 'UPSTAIRS', 'DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


The train dataset contains 7352 records with 128 length (# of data points) and 9 components (# of sensors).
The test dataset contains 2947 records with 128 length (# of data points) and 9 components (# of sensors).


In [5]:
# Define the feature extraction method for each subband

def calculate_statistics(list_values):
    median = np.nanpercentile(list_values, 50)
    mean = np.nanmean(list_values)
    std = np.nanstd(list_values)
    var = np.nanvar(list_values)
    return [median, mean, std, var]


# Define feature extraction methods
def get_features(list_values):
    statistics = calculate_statistics(list_values)
    return statistics

def get_uci_har_features(dataset, labels, waveletname, waveletlevel):
    uci_har_features = []
    for signal_no in range(0, dataset.shape[0]):

        if ((signal_no % 500) == 0):
            print('get_uci_har_features, loop %d/%d' % (signal_no, len(dataset)))
        features = []

        for signal_comp in range(0, dataset.shape[2]): # 9 components
            signal = dataset[signal_no, :, signal_comp]
            list_coeff = pywt.wavedec(signal, waveletname, level=waveletlevel)

            for coeff in list_coeff:
                features += get_features(coeff) # append two lists

        uci_har_features.append(features)
    X = np.array(uci_har_features)
    Y = np.array(labels)
    return X, Y


In [6]:
# Extract features for both train and test signals
waveletname = 'db4'
waveletlevel = 3

print('Generate features for training data')
X_train, Y_train = get_uci_har_features(uci_har_signals_train, uci_har_labels_train, waveletname, waveletlevel)

print('Generate features for training data')
X_test, Y_test = get_uci_har_features(uci_har_signals_test, uci_har_labels_test, waveletname, waveletlevel)

print('X_train shape:', X_train.shape, 'Y_train shape:', Y_train.shape)
print('X_test shape:', X_test.shape, 'Y_test shape:', Y_test.shape)

Generate features for training data
get_uci_har_features, loop 0/7352


get_uci_har_features, loop 500/7352


get_uci_har_features, loop 1000/7352


get_uci_har_features, loop 1500/7352


get_uci_har_features, loop 2000/7352


...

get_uci_har_features, loop 1000/2947


get_uci_har_features, loop 1500/2947


get_uci_har_features, loop 2000/2947


get_uci_har_features, loop 2500/2947


X_train shape: (7352, 144) Y_train shape: (7352,)
X_test shape: (2947, 144) Y_test shape: (2947,)


In [7]:
# Perform classification
clf = GaussianNB()
clf.fit(X_train, Y_train)

Y_predict = clf.predict(X_test)
print(pd.DataFrame(confusion_matrix(Y_test, Y_predict), index=activity_label, columns=activity_label))



            WALKING  UPSTAIRS  DOWNSTAIRS  SITTING  STANDING  LAYING
WALKING         257       163          76        0         0       0
UPSTAIRS         14       437          20        0         0       0
DOWNSTAIRS       83        99         238        0         0       0
SITTING           0         5           0      422        51      13
STANDING          0        20           0      309       189      14
LAYING            0         7           0       40         0     490


$\color{red}{\text{Discussions}}$

Q1: Recommend and describe a method to integrate both time-domain features and wavelet-domain features together to feed them into the classifier model.

Q2: Design an experiment to evaluate the hyperparameters, including (1) the choice of wavelet filters and (2) the wavelet decomposition levels, of the wavelet-based feature extraction for human activity classification.

## Q1: Combining time-domain and wavelet-domain features

Extract complementary time-domain features (mean, median, standard deviation, variance, RMS, minimum, maximum, range and zero-crossing rate) from each sensor. Concatenate these features with the existing wavelet-domain statistics from every DWT subband: `X_combined = [X_time, X_wavelet]`. Fit a scaler on training data only and apply it to both sets. Compare time-only, wavelet-only and fused features with the same classifier. This early-fusion method combines temporal magnitude with multi-resolution frequency information; PCA or feature selection can reduce redundancy if the combined feature vector becomes large.

## Q2: Hyperparameter experiment

Keep the data split, feature statistics, classifier and metrics fixed. Grid-search wavelets `db2`, `db4`, `sym4` and `coif3` with decomposition levels 1, 2, 3 and 4. For each setting, extract training features, fit GaussianNB, and record accuracy, macro-F1, per-class recall and the confusion matrix. Select using a validation split or stratified 5-fold cross-validation on training data, then evaluate the selected setting once on the held-out test set. Report mean and standard deviation across folds and discuss class-specific errors and computational cost.


## AI Tool Declaration

The authors used GPT-5.6 for text translation and assistance in understanding the code. The authors are responsible for the content and quality of the submitted work.

**After you finish the workshop, rename and submit your .ipynb file.**